# Multi-Site-Fanout - clone the reference site into N healthy sites

Replicates the single reference site (`RV*` / *Riverton*) across `N_SITES` synthetic sites
drawn from a site catalog (with real GADS capacity/fuel flavor). Everything is keyed on a
per-site namespace: `RV{u}` -> `{PREFIX}{u}` for tags, asset ids and unit/plant codes, and
the friendly plant name is swapped per site.

**New sites are deliberately healthy:** cloned structure (assets, tags, bridges) + *clean*
`aakr_health` / `predictions_*` (high health, low risk). We do **not** clone `watchlist`,
`anomaly_advisories` or `root_cause`, so nothing is flagged for the new sites - only the
reference site's RV2/RV3 keep their issues. PiEvents history is cloned (normal values,
shifted to ~now) so the tag-driven `PI-Seed-Synthetic` backfill/stream picks the new sites
up automatically.

Idempotent: re-running wipes prior fan-out rows first (Delta rows whose key starts with a
site prefix; PiEvents rows with `Source == 'synthetic-fanout'`).

Run order in deploy: after `_load_data` + PiEvents ingest, before the seed.


In [ ]:
# PARAMETERS  (Fabric: this cell is tagged 'parameters')
WS         = "163ba38c-3869-406f-adb7-37cbc981390c"   # rebound to target workspace id
LH         = "7e08480c-cf8d-4206-901d-38b74dbe35d9"   # rebound to target lakehouse id
KUSTO_URI  = "https://trd-8a08ckb2duw406mvvg.z2.kusto.fabric.microsoft.com"  # host rebound to target eventhouse
DATABASE   = "pi-realtime-db"
PI_TABLE   = "PiEvents"

N_SITES             = 8            # how many synthetic sites to add (capped at len(SITE_CATALOG))
SEED                = 42
REF_ROOT            = "RV"         # reference site's root token (RV1/RV2/RV3 ...)
REF_PLANT_NAME      = "Riverton"   # reference friendly plant name in gold.dim_asset.plant
PIEVENTS_CLONE_DAYS = 5            # clone this many days of the reference tail per site
FANOUT_SOURCE       = "synthetic-fanout"   # Source stamp on cloned PiEvents (idempotent delete key)
CLONE_PIEVENTS      = True         # set False to fan out only the Delta model (no Eventhouse writes)


In [ ]:
# Setup: spark helpers + lakehouse paths
import re, json, ssl, time, random, urllib.request, datetime as dt
from functools import reduce
from pyspark.sql import functions as F, Window, DataFrame
from delta.tables import DeltaTable

BASE   = f"abfss://{WS}@onelake.dfs.fabric.microsoft.com/{LH}"
TABLES = f"{BASE}/Tables"
random.seed(SEED)

def tpath(schema, table): return f"{TABLES}/{schema}/{table}"
def read(schema, table): return spark.read.format("delta").load(tpath(schema, table))
def has(schema, table):
    try:
        read(schema, table).limit(1).count(); return True
    except Exception:
        return False

print("Lakehouse:", BASE)


In [ ]:
# Site catalog: (friendly name, PI/asset prefix). First N_SITES are used.
SITE_CATALOG = [
    ("Ashford", "AS"), ("Brookline", "BK"), ("Cedar Falls", "CF"), ("Deepwater", "DW"),
    ("Eastport", "EP"), ("Fairview", "FV"), ("Glenwood", "GW"), ("Harbor Point", "HP"),
    ("Ironwood", "IW"), ("Junction", "JC"), ("Kingsport", "KP"), ("Lakemont", "LS"),
]
N = max(0, min(int(N_SITES), len(SITE_CATALOG)))

# Optional realistic flavor (capacity / fuel) from the real GADS unit registry, if present.
FUEL = {"GG": "Gas", "DI": "Diesel", "NU": "Nuclear", "BIT": "Coal", "OI": "Oil",
        "WA": "Hydro", "WD": "Wood", "SU": "Solar", "WND": "Wind", "NG": "Gas"}
gads = None
try:
    g = (read("dbo", "gads_unit_configuration")
         .select("MAX_DESIGN_CAP", "PRIMARY_FUEL")
         .where("MAX_DESIGN_CAP is not null").limit(200).toPandas())
    if len(g):
        gads = g.sample(frac=1, random_state=SEED).reset_index(drop=True)
except Exception as e:
    print("  gads flavor unavailable:", str(e)[:80])

SITES = []
for i in range(N):
    name, prefix = SITE_CATALOG[i]
    cap, fuel = None, None
    if gads is not None and i < len(gads):
        try:
            cap = int(gads.loc[i, "MAX_DESIGN_CAP"])
            fuel = FUEL.get(str(gads.loc[i, "PRIMARY_FUEL"]).strip(), str(gads.loc[i, "PRIMARY_FUEL"]).strip())
        except Exception:
            pass
    SITES.append({"prefix": prefix, "plant_name": name, "cap_mw": cap, "fuel": fuel})

PREFIXES  = [s["prefix"] for s in SITES]
PREFIX_RE = ("^(" + "|".join(PREFIXES) + ")[0-9]") if PREFIXES else "^__none__"
print(f"Fanning out {N} site(s):",
      ", ".join(f"{s['plant_name']}({s['prefix']}"
                + (f", {s['cap_mw']}MW {s['fuel']}" if s['cap_mw'] else "") + ")" for s in SITES))


In [ ]:
# Remap helpers: RV{u} -> {PREFIX}{u} everywhere; friendly plant name swapped separately.
def remap_ids(col, prefix):
    return F.regexp_replace(F.col(col), REF_ROOT + r"([0-9])", prefix + r"$1")

def fan_structure(df, id_cols, friendly_plant_col=None, extra=None):
    frames = []
    for s in SITES:
        d = df
        for c in id_cols:
            if c in d.columns:
                d = d.withColumn(c, remap_ids(c, s["prefix"]))
        if friendly_plant_col and friendly_plant_col in d.columns:
            d = d.withColumn(friendly_plant_col,
                    F.when(F.col(friendly_plant_col) == F.lit(REF_PLANT_NAME), F.lit(s["plant_name"]))
                     .otherwise(F.regexp_replace(F.col(friendly_plant_col), REF_ROOT + r"([0-9])", s["prefix"] + r"$1")))
        if extra:
            d = extra(d, s)
        frames.append(d)
    return reduce(DataFrame.unionByName, frames) if frames else None

def wipe_fanout(schema, table, key_col):
    try:
        DeltaTable.forPath(spark, tpath(schema, table)).delete(F.col(key_col).rlike(PREFIX_RE))
    except Exception as e:
        print(f"  wipe {schema}.{table}: {str(e)[:90]}")

def append(schema, table, df, order_cols):
    (df.select(order_cols).write.format("delta").mode("append")
       .option("mergeSchema", "false").save(tpath(schema, table)))

def ref_only(df, key_col):
    return df.where(~F.col(key_col).rlike(PREFIX_RE))


In [ ]:
# ---- STRUCTURE fan-out: clone the reference site's shape onto every new site ----
if N == 0:
    print("N_SITES = 0 -> nothing to fan out")
else:
    STRUCTURE = [
        # schema, table, id_cols (RV{u} remapped), friendly_plant_col, wipe key
        ("gold", "dim_asset",                ["asset_id", "asset_display_name", "running_tag", "unit"], "plant", "asset_id"),
        ("gold", "dim_scope_asset",          ["asset_id", "plant"],                None, "asset_id"),
        ("gold", "running_indicator",        ["asset_id", "Tag", "tag_description"], None, "asset_id"),
        ("dbo",  "pi_tags_metadata",         ["Tag", "Name", "Plant"],             None, "Tag"),
        ("dbo",  "bridge_pi_tag_to_asset",   ["Tag", "asset_id"],                  None, "asset_id"),
        ("gold", "bridge_pi_tag_to_asset",   ["Tag", "asset_id"],                  None, "asset_id"),
        ("dbo",  "bridge_pi_tag_to_equipment", ["Tag", "Plant", "equip_prefix"],   None, "Tag"),
        ("ml",   "selected_tags",            ["tag_asset_id", "Tag"],              None, "tag_asset_id"),
    ]
    for schema, table, id_cols, fpc, key in STRUCTURE:
        if not has(schema, table):
            print(f"  skip {schema}.{table} (missing)"); continue
        base = ref_only(read(schema, table), key)
        cols = base.columns
        out  = fan_structure(base, id_cols, fpc)
        wipe_fanout(schema, table, key)
        append(schema, table, out, cols)
        print(f"  {schema}.{table}: +{out.count():,} rows across {N} site(s)")


In [ ]:
# ---- CLEAN ml fan-out: healthy health + low-risk predictions for new assets ----
# (No watchlist / anomaly_advisories / root_cause -> new sites are never flagged.)
if N > 0:
    now_ts = F.current_timestamp()

    def one_per_asset(df, key="asset_id"):
        w = Window.partitionBy(key).orderBy(F.lit(1))
        return df.withColumn("_rn", F.row_number().over(w)).where(F.col("_rn") == 1).drop("_rn")

    # aakr_health -> high health, no anomalies
    if has("ml", "aakr_health"):
        base = one_per_asset(ref_only(read("ml", "aakr_health"), "asset_id"))
        cols = base.columns
        def health_over(d, s):
            d = (d.withColumn("health_score", F.round(F.lit(96.0) + F.rand(SEED) * 3.4, 2))
                  .withColumn("anomaly_pct", F.lit(0.0))
                  .withColumn("mean_abs_z", F.round(F.lit(0.30) + F.rand(SEED + 1) * 0.5, 3))
                  .withColumn("max_abs_z", F.round(F.lit(1.00) + F.rand(SEED + 2) * 0.8, 3))
                  .withColumn("scored_at", F.date_format(now_ts, "yyyy-MM-dd'T'HH:mm:ss"))
                  .withColumn("notebook_run_id", F.lit("multisite-fanout")))
            if "anomalous_tag_bins" in d.columns:
                d = d.withColumn("anomalous_tag_bins", F.lit(0).cast("long"))
            return d
        out = fan_structure(base, ["asset_id"], None, health_over)
        wipe_fanout("ml", "aakr_health", "asset_id"); append("ml", "aakr_health", out, cols)
        print(f"  ml.aakr_health: +{out.count()} healthy rows")

    # predictions_shortterm -> Normal, tiny stop probability
    if has("ml", "predictions_shortterm"):
        base = one_per_asset(ref_only(read("ml", "predictions_shortterm"), "asset_id"))
        cols = base.columns
        def ps_over(d, s):
            return (d.withColumn("stop_probability", F.round(F.lit(0.01) + F.rand(SEED + 3) * 0.03, 4))
                     .withColumn("alert_level", F.lit("Normal"))
                     .withColumn("scoring_timestamp", now_ts)
                     .withColumn("scored_at", now_ts)
                     .withColumn("model_run_id", F.lit("multisite-fanout")))
        out = fan_structure(base, ["asset_id"], None, ps_over)
        wipe_fanout("ml", "predictions_shortterm", "asset_id"); append("ml", "predictions_shortterm", out, cols)
        print(f"  ml.predictions_shortterm: +{out.count()} clean rows")

    # predictions_longterm -> Low risk, high survival
    if has("ml", "predictions_longterm"):
        base = one_per_asset(ref_only(read("ml", "predictions_longterm"), "asset_id"))
        cols = base.columns
        def pl_over(d, s):
            d = (d.withColumn("risk_score", F.round(F.lit(4.0) + F.rand(SEED + 4) * 10.0, 2))
                  .withColumn("survival_probability_7d", F.round(F.lit(0.985) + F.rand(SEED + 5) * 0.01, 4))
                  .withColumn("survival_probability_14d", F.round(F.lit(0.970) + F.rand(SEED + 6) * 0.02, 4))
                  .withColumn("predicted_median_survival_days", F.round(F.lit(380.0) + F.rand(SEED + 7) * 160.0, 0))
                  .withColumn("risk_level", F.lit("Low"))
                  .withColumn("scoring_timestamp", now_ts)
                  .withColumn("model_run_timestamp", now_ts)
                  .withColumn("model_run_id", F.lit("multisite-fanout")))
            if "scoring_date" in d.columns:
                d = d.withColumn("scoring_date", F.current_date())
            return d
        out = fan_structure(base, ["asset_id"], None, pl_over)
        wipe_fanout("ml", "predictions_longterm", "asset_id"); append("ml", "predictions_longterm", out, cols)
        print(f"  ml.predictions_longterm: +{out.count()} low-risk rows")


In [ ]:
# ---- PiEvents fan-out (Eventhouse / KQL): clone the reference tail per site ----
# Pure server-side KQL (.append) -- no Spark Kusto connector needed (that jar isn't in the
# default Fabric runtime). Cloned rows carry Source='synthetic-fanout', tag/plant remapped
# RV{u}->{PREFIX}{u}, and Ts shifted so the newest point ~ now. PI-Seed-Synthetic then
# backfills the gap and streams live for every new tag automatically.
if N > 0 and CLONE_PIEVENTS:
    try:
        TOKEN = kusto_token()
    except Exception as e:
        TOKEN = None; print("  PiEvents clone skipped - no Kusto token:", str(e)[:100])

    if TOKEN:
        # 1) idempotent: drop any prior fan-out rows
        try:
            kusto_mgmt(f".delete table {PI_TABLE} records <| {PI_TABLE} | where Source == '{FANOUT_SOURCE}'", TOKEN)
            print("  cleared prior fan-out PiEvents rows")
        except Exception as e:
            print("  clear prior fan-out skipped:", str(e)[:110])

        # 2) one server-side .append per site (remap RV{u}->{prefix}{u}, time-shift to ~now)
        for s in SITES:
            pfx = s["prefix"]
            csl = (
                f".append {PI_TABLE} <|\n"
                f"let FANOUT = '{FANOUT_SOURCE}';\n"
                f"let maxTs = toscalar({PI_TABLE} | where Source != FANOUT | summarize max(Ts));\n"
                f"let offset = now() - maxTs;\n"
                f"{PI_TABLE}\n"
                f"| where Source != FANOUT and Ts > maxTs - {int(PIEVENTS_CLONE_DAYS)}d and isnotempty(Tag)\n"
                f"| extend Tag = replace_regex(Tag, @'{REF_ROOT}([0-9])', @'{pfx}\\1'),\n"
                f"         Plant = replace_regex(coalesce(Plant, ''), @'{REF_ROOT}([0-9])', @'{pfx}\\1'),\n"
                f"         Ts = Ts + offset, Source = FANOUT, Host = 'FANOUT', WebId = '', IngestedAt = now()\n"
                f"| project Ts, WebId, Tag, Plant, Value, ValueType, Questionable, Substituted, Source, Host, IngestedAt"
            )
            try:
                kusto_mgmt(csl, TOKEN)
                print(f"    {s['plant_name']} ({pfx}): appended ~{int(PIEVENTS_CLONE_DAYS)}d reference tail")
            except Exception as e:
                print(f"    {s['plant_name']} ({pfx}) append FAILED:", str(e)[:160])
else:
    print("  PiEvents clone disabled (CLONE_PIEVENTS=False or N_SITES=0)")

In [ ]:
# ---- VERIFY ----
print("=== VERIFY ===")
try:
    plants = sorted(r["plant"] for r in read("gold", "dim_asset").select("plant").distinct().collect())
    print(f"  gold.dim_asset plants ({len(plants)}):", plants)
    print("  gold.dim_asset total assets:", read("gold", "dim_asset").count())
except Exception as e:
    print("  dim_asset:", str(e)[:100])
for sch, tab, key in [("ml", "aakr_health", "asset_id"),
                      ("ml", "predictions_shortterm", "asset_id"),
                      ("ml", "predictions_longterm", "asset_id")]:
    if has(sch, tab):
        df = read(sch, tab)
        print(f"  {sch}.{tab}: total={df.count()}, fan-out={df.where(F.col(key).rlike(PREFIX_RE)).count()}")
print("  (watchlist / anomaly_advisories / root_cause intentionally NOT fanned out -> new sites unflagged)")
